In [2]:
import os
import nibabel as nib
import numpy as np
import cv2

INPUT_PATH = "OASIS"
OUTPUT_PATH = "dataset"

os.makedirs(OUTPUT_PATH, exist_ok=True)

for root, dirs, files in os.walk(INPUT_PATH):

    for file in files:

        if file.endswith(".nii"):

            path = os.path.join(root, file)

            img = nib.load(path)
            data = img.get_fdata()

            mid_slice = data[:,:,data.shape[2]//2]

            mid_slice = cv2.normalize(mid_slice,None,0,255,cv2.NORM_MINMAX)
            mid_slice = mid_slice.astype(np.uint8)

            save_path = os.path.join(OUTPUT_PATH, file.replace(".nii",".jpg"))

            cv2.imwrite(save_path, mid_slice)

print("Conversion completed")

Conversion completed


In [3]:
import os

dataset = "dataset_combined"

for root, dirs, files in os.walk(dataset):
    images = [f for f in files if f.lower().endswith(('.png','.jpg','.jpeg'))]
    if images:
        print(root, ":", len(images), "images")

dataset_combined\Alzheimer : 19734 images
dataset_combined\MCI : 20684 images
dataset_combined\Normal : 25265 images


In [4]:
import os
import shutil

dataset = "dataset_combined"

for class_folder in os.listdir(dataset):
    class_path = os.path.join(dataset, class_folder)

    for subfolder in os.listdir(class_path):
        sub_path = os.path.join(class_path, subfolder)

        if os.path.isdir(sub_path):
            for file in os.listdir(sub_path):
                src = os.path.join(sub_path, file)
                dst = os.path.join(class_path, file)
                shutil.move(src, dst)

            os.rmdir(sub_path)

print("Dataset flattened successfully")

Dataset flattened successfully


In [7]:
import os
import shutil
import random

source = "dataset_combined"
train_dir = "dataset/train"
test_dir = "dataset/test"

classes = ["Alzheimer","MCI","Normal"]

for c in classes:
    os.makedirs(os.path.join(train_dir,c),exist_ok=True)
    os.makedirs(os.path.join(test_dir,c),exist_ok=True)

    images = os.listdir(os.path.join(source,c))
    random.shuffle(images)

    split = int(0.8*len(images))

    train_imgs = images[:split]
    test_imgs = images[split:]

    for img in train_imgs:
        shutil.copy(os.path.join(source,c,img),
                    os.path.join(train_dir,c,img))

    for img in test_imgs:
        shutil.copy(os.path.join(source,c,img),
                    os.path.join(test_dir,c,img))

print("Train-Test split completed")

Train-Test split completed


In [10]:
import os
import cv2
import numpy as np

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model

dataset = "dataset/train"

classes = ["Alzheimer", "MCI", "Normal"]

MAX_IMAGES = 500

# containers for features and labels
data = []
labels = []

# load VGG16 feature extractor
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224,224,3))
model = Model(inputs=base_model.input, outputs=base_model.output)

for label in classes:
    path = os.path.join(dataset, label)

    imgs = os.listdir(path)[:MAX_IMAGES]

    print(label, "processing", len(imgs), "images")

    for img in imgs:

        img_path = os.path.join(path, img)

        image = cv2.imread(img_path)

        if image is None:
            continue

        image = cv2.resize(image,(224,224))

        image = np.expand_dims(image, axis=0)

        image = preprocess_input(image)

        features = model.predict(image)

        features = features.flatten()

        data.append(features)
        labels.append(label)

print("Total samples:", len(data))

Alzheimer processing 500 images
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 555ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159m

In [11]:
# convert lists to numpy arrays
X = np.array(data)
y = np.array(labels)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# save them
np.save("features.npy", X)
np.save("labels.npy", y)

print("Features and labels saved")

Feature shape: (1500, 25088)
Label shape: (1500,)
Features and labels saved


In [ ]:
# convert lists to numpy arrays
X = np.array(data)
y = np.array(labels)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# save them
np.save("features.npy", X)
np.save("labels.npy", y)

print("Features and labels saved")

In [12]:
count = 0
MAX_IMAGES = 1000

for label in classes:
    path = os.path.join(dataset,label)

    for img in os.listdir(path):
        if count >= MAX_IMAGES:
            break

        # preprocessing
        count += 1

In [1]:
import os
import cv2
import numpy as np
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

dataset = "dataset/train"

labels = []
data = []

classes = ["Alzheimer","MCI","Normal"]

model = VGG16(weights="imagenet",include_top=False,input_shape=(224,224,3))

for label in classes:

    path = os.path.join(dataset,label)

    for img in os.listdir(path):

        img_path = os.path.join(path,img)

        image = cv2.imread(img_path)
        image = cv2.resize(image,(224,224))

        image = preprocess_input(image)

        features = model.predict(np.expand_dims(image,axis=0))
        features = features.flatten()

        data.append(features)
        labels.append(label)

data = np.array(data)

X_train,X_test,y_train,y_test = train_test_split(data,labels,test_size=0.2)

clf = SVC(kernel='linear')

clf.fit(X_train,y_train)

pred = clf.predict(X_test)

print(classification_report(y_test,pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 529ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

KeyboardInterrupt: 